<a href="https://colab.research.google.com/github/santiagosernab76-boop/Programacion-/blob/main/Proyecto_final_programacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control de gastos personales
**Proyecto final – ADMI 1301 Introducción a la Programación** · Universidad de los Andes · 2026-2
**Estudiante:** Santiago Serna (202414950) · **Profesor:** Gustavo Niño

## ¿Qué hace?
Es una aplicación para registrar gastos (descripción, valor, categoría y fecha), verlos por categoría o por mes y compararlos con un **presupuesto mensual**. Está pensada para estudiantes que quieren saber en qué se les va la plata.

## ¿Cómo se usa?
1. Ejecuta las celdas **en orden** (`Shift + Enter`).
2. La aplicación **ya trae 10 gastos de prueba** (ficticios) y un presupuesto de $600.000, no hay que cargar nada.
3. La última celda abre el menú. Escribe el número de la opción en el cuadro que aparece abajo.

⚠ Los datos están **solo en memoria** (en una lista). Si se reinicia el notebook, se pierden.

Cada gasto es una lista: `[descripcion, valor, categoria, fecha]`, y todos están dentro de la lista `gastos`.

In [10]:
from datetime import datetime   # solo lo uso para revisar las fechas

# categorías que se pueden usar
categorias = ["Alimentación", "Transporte", "Entretenimiento", "Compras"]

# lista principal: cada gasto es [descripcion, valor, categoria, fecha]
# estos son datos de prueba (inventados) para poder probar el programa
gastos = [
    ["Mercado del mes",           95000, "Alimentación",    "2026-10-20"],
    ["Recarga de transporte",     40000, "Transporte",      "2026-10-22"],
    ["Almuerzo en la cafetería",  18000, "Alimentación",    "2026-11-02"],
    ["Recarga de transporte",     50000, "Transporte",      "2026-11-03"],
    ["Cine con amigos",           32000, "Entretenimiento", "2026-11-05"],
    ["Mercado de la semana",     120000, "Alimentación",    "2026-11-08"],
    ["Cuaderno y útiles",         45000, "Compras",         "2026-11-10"],
    ["Taxi",                      22000, "Transporte",      "2026-11-12"],
    ["Suscripción streaming",     29900, "Entretenimiento", "2026-11-15"],
    ["Zapatos",                  150000, "Compras",         "2026-11-18"],
]

presupuesto = 600000   # presupuesto mensual (0 = no hay presupuesto)

## 1. Funciones para revisar lo que escribe el usuario
Cada función `revisar_...` recibe el texto y devuelve **(dato, error)**. Si todo está bien el error es `""`; si no, el error dice cómo corregirlo. Las funciones `pedir_...` usan un `while` para repetir la pregunta hasta que el dato esté bien, así el programa no se cae.

In [11]:
def dinero(valor):
    # pone el valor así: $1.234.567
    if valor == int(valor):
        texto = f"{int(valor):,}"
    else:
        texto = f"{valor:,.2f}"
    # cambio la coma por punto y el punto por coma
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    return "$" + texto


def revisar_valor(texto):
    texto = texto.strip()
    if texto == "":
        return 0, "El valor no puede estar vacío. Escribe un número, por ejemplo 15000."
    if texto.startswith("-"):
        return 0, "El valor no puede ser negativo. Escribe un número mayor que cero."
    # solo dejo números con máximo un punto decimal
    if not (texto.isascii() and texto.replace(".", "", 1).isdigit()):
        return 0, "Formato incorrecto. Escribe solo números, sin $ ni comas. Ejemplo: 15000 o 15000.50"
    valor = round(float(texto), 2)
    if valor == 0:
        return 0, "El valor no puede ser cero. Escribe un número mayor que cero."
    return valor, ""


def revisar_categoria(texto):
    texto = texto.strip().lower()
    if texto == "":
        return "", "La categoría no puede estar vacía. Escribe un número del 1 al 4."
    # si escribe el número
    if texto in ["1", "2", "3", "4"]:
        return categorias[int(texto) - 1], ""
    # si escribe el nombre (con o sin tilde)
    for c in categorias:
        if texto == c.lower() or texto == c.lower().replace("ó", "o"):
            return c, ""
    return "", "Esa categoría no existe. Escribe un número del 1 al 4."


def revisar_fecha(texto):
    texto = texto.strip()
    if texto == "":
        return "", "La fecha no puede estar vacía. Usa el formato AAAA-MM-DD, por ejemplo 2026-11-15."
    try:
        fecha = datetime.strptime(texto, "%Y-%m-%d")
    except ValueError:
        return "", "Fecha inválida. Usa el formato AAAA-MM-DD con una fecha real, por ejemplo 2026-11-15."
    return fecha.strftime("%Y-%m-%d"), ""


def revisar_mes(texto):
    texto = texto.strip()
    if texto == "":
        return "", "El mes no puede estar vacío. Usa el formato AAAA-MM, por ejemplo 2026-11."
    try:
        fecha = datetime.strptime(texto, "%Y-%m")
    except ValueError:
        return "", "Mes inválido. Usa el formato AAAA-MM, por ejemplo 2026-11."
    return fecha.strftime("%Y-%m"), ""


def pedir_valor(mensaje):
    while True:
        valor, error = revisar_valor(input(mensaje))
        if error == "":
            return valor
        print("   ⚠", error)


def pedir_categoria():
    print("Categorías: 1. Alimentación  2. Transporte  3. Entretenimiento  4. Compras")
    while True:
        cat, error = revisar_categoria(input("Categoría (número o nombre): "))
        if error == "":
            return cat
        print("   ⚠", error)


def pedir_fecha():
    while True:
        fecha, error = revisar_fecha(input("Fecha (AAAA-MM-DD): "))
        if error == "":
            return fecha
        print("   ⚠", error)


def pedir_mes():
    while True:
        mes, error = revisar_mes(input("Mes (AAAA-MM, ejemplo 2026-11): "))
        if error == "":
            return mes
        print("   ⚠", error)

## 2. Cálculos
Todas reciben una **lista de gastos** y la recorren con un `for`. Como reciben la lista, sirven para cualquier periodo: si les paso solo los gastos de un mes, calculan solo ese mes.

- `total()` suma todos los valores.
- `total_categoria()` suma solo los de una categoría.
- `categoria_mayor()` busca la categoría donde más se gastó.
- `estado_presupuesto()` compara lo gastado con el presupuesto (disponible, alcanzado o superado).

In [12]:
def filtrar_categoria(lista, categoria):
    resultado = []
    for g in lista:
        if g[2] == categoria:
            resultado.append(g)
    return resultado


def filtrar_mes(lista, mes):
    resultado = []
    for g in lista:
        if g[3][:7] == mes:      # los primeros 7 caracteres de la fecha son el mes
            resultado.append(g)
    return resultado


def total(lista):
    suma = 0
    for g in lista:
        suma = suma + g[1]
    return round(suma, 2)


def total_categoria(lista, categoria):
    suma = 0
    for g in lista:
        if g[2] == categoria:
            suma = suma + g[1]
    return round(suma, 2)


def categoria_mayor(lista):
    mayor = 0
    nombre = "Ninguna"
    for c in categorias:
        t = total_categoria(lista, c)
        if t > mayor:
            mayor = t
            nombre = c
    return nombre, mayor


def estado_presupuesto(gastado, presupuesto):
    # devuelve: estado, dinero (lo que queda o lo que se pasó) y porcentaje usado
    gastado = round(gastado, 2)
    presupuesto = round(presupuesto, 2)
    porcentaje = round(gastado / presupuesto * 100, 2)
    if gastado < presupuesto:
        return "disponible", round(presupuesto - gastado, 2), porcentaje
    elif gastado == presupuesto:
        return "alcanzado", 0, 100.0
    else:
        return "superado", round(gastado - presupuesto, 2), porcentaje

## 3. Opciones del menú
- **Registrar gasto:** pide los 4 datos (cada uno revisado) y lo agrega a la lista con `append`. Un `while` deja registrar varios gastos seguidos.
- **Consultar por categoría / por mes:** filtra la lista y muestra una tabla con los gastos y el total. En la consulta por categoría también se puede elegir ver todos los gastos.
- **Establecer o modificar presupuesto:** guarda el valor en la variable `presupuesto`.
- **Consultar presupuesto disponible:** pide un mes y muestra el resumen: presupuesto, gastado, cuánto queda (o cuánto se pasó), porcentaje usado, cantidad de gastos, promedio, categoría con más gasto y el gasto por categoría.

In [13]:
def mostrar_lista(lista, titulo):
    print()
    print("=" * 74)
    print(titulo)
    print("=" * 74)
    if len(lista) == 0:
        print("No hay gastos para mostrar.")
        return
    print(f"{'N°':<4}{'Fecha':<12}{'Descripción':<28}{'Categoría':<17}{'Valor':>12}")
    print("-" * 74)
    n = 1
    for g in lista:
        print(f"{n:<4}{g[3]:<12}{g[0]:<28}{g[2]:<17}{dinero(g[1]):>12}")
        n = n + 1
    print("-" * 74)
    print(f"{len(lista)} gasto(s). Total: {dinero(total(lista))}")


def registrar_gasto():
    seguir = "s"
    while seguir == "s":
        print("\n--- Registrar un gasto ---")
        descripcion = input("Descripción: ").strip()
        while descripcion == "" or len(descripcion) > 28:
            print("   ⚠ La descripción no puede estar vacía ni tener más de 28 letras.")
            descripcion = input("Descripción: ").strip()
        valor = pedir_valor("Valor (solo números, ejemplo 15000): ")
        categoria = pedir_categoria()
        fecha = pedir_fecha()

        gastos.append([descripcion, valor, categoria, fecha])
        print(f"✔ Gasto registrado: {descripcion} | {dinero(valor)} | {categoria} | {fecha}")

        seguir = input("¿Registrar otro gasto? (s/n): ").strip().lower()
        while seguir != "s" and seguir != "n":
            seguir = input("Responde solo s o n: ").strip().lower()


def consultar_por_categoria():
    print("\n--- Consultar gastos por categoría ---")
    print("Categorías: 1. Alimentación  2. Transporte  3. Entretenimiento  4. Compras  5. Todas")
    texto = input("Categoría (número o nombre): ").strip()
    if texto == "5" or texto.lower() == "todas":
        mostrar_lista(gastos, "TODOS LOS GASTOS")
        return
    cat, error = revisar_categoria(texto)
    while error != "":
        print("   ⚠", error)
        texto = input("Categoría (número o nombre): ")
        cat, error = revisar_categoria(texto)
    mostrar_lista(filtrar_categoria(gastos, cat), "GASTOS DE " + cat.upper())


def consultar_por_mes():
    print("\n--- Consultar gastos por mes ---")
    mes = pedir_mes()
    lista = filtrar_mes(gastos, mes)
    if len(lista) == 0:
        print(f"No hay gastos registrados en {mes}.")
    else:
        mostrar_lista(lista, "GASTOS DEL MES " + mes)


def establecer_presupuesto():
    global presupuesto            # uso global porque cambio la variable de afuera
    print("\n--- Establecer o modificar presupuesto ---")
    if presupuesto > 0:
        print("Presupuesto actual:", dinero(presupuesto))
    else:
        print("Todavía no hay presupuesto.")
    nuevo = pedir_valor("Nuevo presupuesto mensual (ejemplo 600000): ")
    presupuesto = nuevo
    print("✔ Presupuesto guardado:", dinero(presupuesto))


def mostrar_presupuesto(lista, mes, presupuesto):
    del_mes = filtrar_mes(lista, mes)
    gastado = total(del_mes)
    print()
    print("=" * 74)
    print("PRESUPUESTO Y RESUMEN DEL MES", mes)
    print("=" * 74)
    if presupuesto <= 0:
        print("Todavía no hay presupuesto. Usa la opción 4 para establecerlo.")
        return
    estado, dinero_estado, porcentaje = estado_presupuesto(gastado, presupuesto)
    print("Presupuesto mensual:    ", dinero(presupuesto))
    print("Total gastado:          ", dinero(gastado))
    if estado == "disponible":
        print("Dinero disponible:      ", dinero(dinero_estado))
        print("Porcentaje utilizado:   ", str(porcentaje) + "%")
        print("✔ Todavía tienes presupuesto disponible.")
    elif estado == "alcanzado":
        print("Porcentaje utilizado:    100%")
        print("⚠ Presupuesto alcanzado: ya usaste el 100%.")
    else:
        exceso = round(dinero_estado / presupuesto * 100, 2)
        print("Dinero excedido:        ", dinero(dinero_estado))
        print("Porcentaje utilizado:   ", str(porcentaje) + "%")
        print(f"✘ Presupuesto superado: te pasaste por {dinero(dinero_estado)} ({exceso}% más del presupuesto).")
    print("-" * 74)
    print("Cantidad de gastos:     ", len(del_mes))
    if len(del_mes) > 0:
        print("Promedio por gasto:     ", dinero(round(gastado / len(del_mes), 2)))
        nombre, valor = categoria_mayor(del_mes)
        print("Categoría con más gasto: ", nombre, "(" + dinero(valor) + ")")
        print("Gasto por categoría:")
        for c in categorias:
            print(f"   {c:<17}{dinero(total_categoria(del_mes, c)):>12}")
    else:
        print("No hay gastos registrados en ese mes.")


def consultar_presupuesto():
    print("\n--- Consultar presupuesto disponible ---")
    mes = pedir_mes()
    mostrar_presupuesto(gastos, mes, presupuesto)

## 4. Pruebas rápidas
Comparo lo que calcula el programa con lo que calculé a mano para noviembre de 2026:
Alimentación = 18.000 + 120.000 = 138.000 · Transporte = 50.000 + 22.000 = 72.000 · Entretenimiento = 32.000 + 29.900 = 61.900 · Compras = 45.000 + 150.000 = 195.000 · **Total = 466.900** (8 gastos).
Si algo no coincide, sale un ✘.

In [14]:
correctas = 0
total_pruebas = 0

def probar(nombre, obtenido, esperado):
    global correctas, total_pruebas
    total_pruebas = total_pruebas + 1
    if obtenido == esperado:
        correctas = correctas + 1
        print("✔", nombre)
    else:
        print("✘", nombre, "-> salió", obtenido, "y se esperaba", esperado)

noviembre = filtrar_mes(gastos, "2026-11")

print("Consultas y cálculos")
probar("hay 10 gastos de prueba", len(gastos), 10)
probar("noviembre tiene 8 gastos", len(noviembre), 8)
probar("octubre tiene 2 gastos", len(filtrar_mes(gastos, "2026-10")), 2)
probar("septiembre no tiene gastos", len(filtrar_mes(gastos, "2026-09")), 0)
probar("Alimentación tiene 3 gastos", len(filtrar_categoria(gastos, "Alimentación")), 3)
probar("total de noviembre", total(noviembre), 466900)
probar("Alimentación noviembre", total_categoria(noviembre, "Alimentación"), 138000)
probar("Transporte noviembre", total_categoria(noviembre, "Transporte"), 72000)
probar("Entretenimiento noviembre", total_categoria(noviembre, "Entretenimiento"), 61900)
probar("Compras noviembre", total_categoria(noviembre, "Compras"), 195000)
probar("categoría con más gasto", categoria_mayor(noviembre), ("Compras", 195000))
probar("total de un mes sin gastos", total([]), 0)

print("\nPresupuesto (gastado en noviembre = 466900)")
probar("presupuesto 600000 -> disponible", estado_presupuesto(466900, 600000), ("disponible", 133100, 77.82))
probar("presupuesto 466900 -> alcanzado", estado_presupuesto(466900, 466900), ("alcanzado", 0, 100.0))
probar("presupuesto 350000 -> superado", estado_presupuesto(466900, 350000), ("superado", 116900, 133.4))
probar("0.1 + 0.2 contra 0.3 -> alcanzado", estado_presupuesto(0.1 + 0.2, 0.3)[0], "alcanzado")

print("\nDatos incorrectos (tienen que dar error)")
for v in ["-500", "0", "", "abc", "$5000", "15.000,50"]:
    probar("valor " + repr(v) + " se rechaza", revisar_valor(v)[1] != "", True)
for c in ["Salud", "9", ""]:
    probar("categoría " + repr(c) + " se rechaza", revisar_categoria(c)[1] != "", True)
for f in ["", "2026-02-30", "ayer", "32/13/2026"]:
    probar("fecha " + repr(f) + " se rechaza", revisar_fecha(f)[1] != "", True)
for m in ["", "2026-13", "noviembre"]:
    probar("mes " + repr(m) + " se rechaza", revisar_mes(m)[1] != "", True)
probar("valor 15000.50 se acepta", revisar_valor("15000.50"), (15000.5, ""))
probar("categoría '3' es Entretenimiento", revisar_categoria("3")[0], "Entretenimiento")

print("\nResultado:", correctas, "de", total_pruebas, "pruebas correctas")

Consultas y cálculos
✔ hay 10 gastos de prueba
✔ noviembre tiene 8 gastos
✔ octubre tiene 2 gastos
✔ septiembre no tiene gastos
✔ Alimentación tiene 3 gastos
✔ total de noviembre
✔ Alimentación noviembre
✔ Transporte noviembre
✔ Entretenimiento noviembre
✔ Compras noviembre
✔ categoría con más gasto
✔ total de un mes sin gastos

Presupuesto (gastado en noviembre = 466900)
✔ presupuesto 600000 -> disponible
✔ presupuesto 466900 -> alcanzado
✔ presupuesto 350000 -> superado
✔ 0.1 + 0.2 contra 0.3 -> alcanzado

Datos incorrectos (tienen que dar error)
✔ valor '-500' se rechaza
✔ valor '0' se rechaza
✔ valor '' se rechaza
✔ valor 'abc' se rechaza
✔ valor '$5000' se rechaza
✔ valor '15.000,50' se rechaza
✔ categoría 'Salud' se rechaza
✔ categoría '9' se rechaza
✔ categoría '' se rechaza
✔ fecha '' se rechaza
✔ fecha '2026-02-30' se rechaza
✔ fecha 'ayer' se rechaza
✔ fecha '32/13/2026' se rechaza
✔ mes '' se rechaza
✔ mes '2026-13' se rechaza
✔ mes 'noviembre' se rechaza
✔ valor 15000.50 se

## 5. Ejemplo del resumen (datos de prueba, noviembre de 2026)
Esta celda muestra lo mismo que la opción 5 del menú, sin tener que escribir nada.

In [15]:
print("Tabla 1. Gastos registrados (datos de prueba)")
mostrar_lista(gastos, "TODOS LOS GASTOS")
print("\nTabla 2. Resumen del presupuesto (noviembre de 2026)")
mostrar_presupuesto(gastos, "2026-11", 600000)

Tabla 1. Gastos registrados (datos de prueba)

TODOS LOS GASTOS
N°  Fecha       Descripción                 Categoría               Valor
--------------------------------------------------------------------------
1   2026-10-20  Mercado del mes             Alimentación          $95.000
2   2026-10-22  Recarga de transporte       Transporte            $40.000
3   2026-11-02  Almuerzo en la cafetería    Alimentación          $18.000
4   2026-11-03  Recarga de transporte       Transporte            $50.000
5   2026-11-05  Cine con amigos             Entretenimiento       $32.000
6   2026-11-08  Mercado de la semana        Alimentación         $120.000
7   2026-11-10  Cuaderno y útiles           Compras               $45.000
8   2026-11-12  Taxi                        Transporte            $22.000
9   2026-11-15  Suscripción streaming       Entretenimiento       $29.900
10  2026-11-18  Zapatos                     Compras              $150.000
-----------------------------------------------

## 6. Menú principal
Un `while` muestra el menú, lee la opción y llama a la función. Después de cada opción se vuelve al menú, y solo se sale con la opción 6. Si escribes una opción que no existe, avisa y vuelve a mostrar el menú.

In [16]:
def menu_principal():
    opcion = ""
    while opcion != "6":
        print()
        print("=" * 74)
        print("CONTROL DE GASTOS PERSONALES - MENÚ PRINCIPAL")
        print("=" * 74)
        print("1. Registrar un gasto")
        print("2. Consultar gastos por categoría")
        print("3. Consultar gastos por mes")
        print("4. Establecer o modificar presupuesto")
        print("5. Consultar presupuesto disponible")
        print("6. Salir")
        opcion = input("Elige una opción (1-6): ").strip()

        if opcion == "1":
            registrar_gasto()
        elif opcion == "2":
            consultar_por_categoria()
        elif opcion == "3":
            consultar_por_mes()
        elif opcion == "4":
            establecer_presupuesto()
        elif opcion == "5":
            consultar_presupuesto()
        elif opcion == "6":
            print("Hasta luego. Recuerda que al reiniciar el notebook los datos se pierden.")
        else:
            print("⚠ Opción no válida. Escribe un número del 1 al 6.")

In [ ]:
66

66
66
6# ▶ Ejecuta esta celda para abrir la aplicación
menu_principal()


CONTROL DE GASTOS PERSONALES - MENÚ PRINCIPAL
1. Registrar un gasto
2. Consultar gastos por categoría
3. Consultar gastos por mes
4. Establecer o modificar presupuesto
5. Consultar presupuesto disponible
6. Salir

--- Consultar gastos por mes ---

GASTOS DEL MES 2026-02
N°  Fecha       Descripción                 Categoría               Valor
--------------------------------------------------------------------------
1   2026-02-21  Zapatos                     Compras              $300.000
--------------------------------------------------------------------------
1 gasto(s). Total: $300.000

CONTROL DE GASTOS PERSONALES - MENÚ PRINCIPAL
1. Registrar un gasto
2. Consultar gastos por categoría
3. Consultar gastos por mes
4. Establecer o modificar presupuesto
5. Consultar presupuesto disponible
6. Salir


## Conclusiones
- La aplicación registra gastos, los clasifica en 4 categorías, los consulta por categoría y por mes, y los compara con un presupuesto mensual (disponible, alcanzado o superado).
- Con los datos de prueba, en noviembre de 2026 se gastaron $466.900, la categoría con más gasto fue Compras ($195.000) y con un presupuesto de $600.000 se usó el 77,82% (quedan $133.100).
- **Limitaciones:** los datos solo están en memoria, hay un solo presupuesto para todos los meses, no se pueden editar ni borrar gastos y las categorías son fijas.
- **Conceptos usados:** variables, listas, `if/elif/else`, ciclos `for` y `while`, funciones y validación de entradas (`try/except` solo para las fechas).